# 05 Kalman and DeepAR Hybrid

This experiment tests a meaningful hybrid: causal Kalman state estimates become historical covariates for DeepAR. Forecasts are not simply averaged.

For each day, the Kalman features are calculated **before** observing that day's sales. For future days, the filtered level is projected forward, uncertainty grows with the horizon, and no future actual sales are used.

## 1. Locate the project and import the experiment runner

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root located; Phase 2 modules are importable.")

Project root located; Phase 2 modules are importable.


## 2. Inspect the hybrid features

The three added inputs are:

- filtered local demand level;
- state variance, representing local uncertainty;
- the previous innovation, or recent Kalman forecast error.

They use an expanding past-only scale, so a target can affect only later rows. High-volume items therefore do not dominate the network, and validation or test targets cannot normalize training features.

In [2]:
from src.phase2.experiments import prepare_experiment_data

data, windows, boundaries, maps, feature_set, config = prepare_experiment_data(
    PROJECT_ROOT,
    experiment_name="hybrid",
)

kalman_columns = [name for name in feature_set["dynamic"] if name.startswith("kalman_")]
print("Kalman inputs:", kalman_columns)
display(data[["id", "date", "sales"] + kalman_columns].head(10))

Kalman inputs: ['kalman_level_scaled', 'kalman_variance_scaled', 'kalman_innovation_scaled']


,id,date,sales,kalman_level_scaled,kalman_variance_scaled,kalman_innovation_scaled
0,FOODS_1_026_TX_3_validation,2011-01-29,0,0.000000,1.010000,0.000000
1,FOODS_1_026_TX_3_validation,2011-01-30,0,0.000000,0.210397,0.000000
2,FOODS_1_026_TX_3_validation,2011-01-31,0,0.000000,0.124248,0.000000
3,FOODS_1_026_TX_3_validation,2011-02-01,1,0.000000,0.092998,0.000000
4,FOODS_1_026_TX_3_validation,2011-02-02,0,0.271133,0.084450,1.000000
5,FOODS_1_026_TX_3_validation,2011-02-03,0,0.202671,0.096459,-0.271133
6,FOODS_1_026_TX_3_validation,2011-02-04,0,0.157185,0.099811,-0.202671
7,FOODS_1_026_TX_3_validation,2011-02-05,1,0.112336,0.091332,-0.157185
8,FOODS_1_026_TX_3_validation,2011-02-06,0,0.349853,0.090703,0.887664
9,FOODS_1_026_TX_3_validation,2011-02-07,0,0.256714,0.095127,-0.349853


## 3. Train the hybrid

Architecture and training boundaries remain the same as standalone DeepAR. Therefore, any measured difference is attributable to the added Kalman state information rather than a larger network or a different test population.

In [3]:
from src.phase2.experiments import run_training_experiment

hybrid = run_training_experiment(
    project_root=PROJECT_ROOT,
    experiment_name="hybrid",
    experiment_id="deepar_kalman_hybrid_v2",
    model_name="DeepAR + price + causal Kalman hybrid",
)

display(hybrid["history"])

,epoch,training_loss,validation_loss
0,1,1.253650,1.156058
1,2,1.206869,1.143883
2,3,1.192901,1.145916
3,4,1.187495,1.138011
4,5,1.182211,1.139513
5,6,1.176952,1.137448


## 4. Compare standalone DeepAR and the hybrid on the same 2,800 observations

In [4]:
import pandas as pd

result_dir = PROJECT_ROOT / "results" / "phase2"
base_metrics = pd.read_csv(result_dir / "deepar_base_v2_metrics.csv")
hybrid_metrics = pd.read_csv(result_dir / "deepar_kalman_hybrid_v2_metrics.csv")
comparison = pd.concat([base_metrics, hybrid_metrics], ignore_index=True)

columns = [
    "Model", "MAE", "RMSE", "WAPE_percent", "mean_pinball_loss",
    "interval_coverage_80_percent", "mean_interval_width", "training_seconds",
]
display(comparison[columns])

,Model,MAE,RMSE,WAPE_percent,mean_pinball_loss,interval_coverage_80_percent,mean_interval_width,training_seconds
0,DeepAR + price + availability,0.922143,2.119425,60.454226,0.311310,93.892857,3.371429,6.935905
1,DeepAR + price + causal Kalman hybrid,0.955893,2.272801,62.666823,0.316069,93.250000,3.191893,6.587614


## 5. Interpretation

A hybrid is supported only if the measured metrics improve. Coverage should be read together with interval width: extremely wide intervals can obtain high coverage without being operationally useful.